# Week 3 — Data Preparation

Week 1 established *what* the data is, and Week 2 looked at it: 2,200 rows,
seven numeric features on wildly different scales, 22 crop labels with exactly
100 rows each, no missing values, and no genuine errors among the values the
IQR rule flagged.

This notebook turns that raw table into **model-ready data**, in four steps and
in this order:

1. **Encode the target** — 22 crop *names* become 22 integers, because
   scikit-learn estimators fit on numbers.
2. **Split** — 80% of the rows for training, 20% held back, *stratified* so both
   halves keep 22 crops in equal proportion.
3. **Scale** — a `ColumnTransformer` wrapping a `StandardScaler`, **fitted on
   the training rows only** and then applied to both halves.
4. **Save** — the splits written to `data/processed/` so Week 4 can start from a
   model-ready table instead of repeating all of this.

Step 2 comes before step 3, and that ordering is the entire point of the week.
Scaling first would compute means and standard deviations over rows that are
supposed to be unseen — the **data leakage** Week 2 defined in the abstract and
this notebook makes concrete.

No model is trained here. Week 3 stops at the point where the data is ready;
Week 4 is where a baseline is fitted to it.

The narrative explanation of every technique below lives in
[`docs/curriculum/week03/learning_notes.md`](../docs/curriculum/week03/learning_notes.md).

## 0. Setup

As in Weeks 1 and 2, the first cell puts the repository root on `sys.path` so
`src` is importable from inside `notebooks/`, and the logic itself is imported
rather than written inline: `stratified_split()` lives in `src/data/split.py`
and `build_preprocessor()` in `src/preprocessing/preprocessor.py`, both covered
by `tests/test_preprocessing.py`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

from src.data import (
    DEFAULT_RANDOM_STATE,
    DEFAULT_TEST_SIZE,
    EXPECTED_LABELS,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    class_proportions,
    load_data,
    stratified_split,
)
from src.preprocessing import build_preprocessing_pipeline, build_preprocessor

pd.set_option("display.width", 120)

print("seed", DEFAULT_RANDOM_STATE, "| test_size", DEFAULT_TEST_SIZE)

seed 42 | test_size 0.2


In [2]:
crops = load_data()          # reads the CSV and validates it in one step (Week 1)
FEATURES = list(FEATURE_COLUMNS)
crops.shape

(2200, 8)

## 1. Why raw data cannot go straight into a model

Week 2's summary table is worth restating in the terms that matter now: not
"what does this feature look like?" but "what would an algorithm make of these
numbers as they stand?".

In [3]:
scales = pd.DataFrame(
    {
        "min": crops[FEATURES].min(),
        "max": crops[FEATURES].max(),
        "range": crops[FEATURES].max() - crops[FEATURES].min(),
        "mean": crops[FEATURES].mean(),
        "std": crops[FEATURES].std(),
    }
)
scales["range_vs_ph"] = scales["range"] / scales.loc["ph", "range"]
scales.round(2)

,min,max,range,mean,std,range_vs_ph
N,0.00,140.00,140.00,50.55,36.92,21.77
P,5.00,145.00,140.00,53.36,32.99,21.77
K,5.00,205.00,200.00,48.15,50.65,31.10
temperature,8.83,43.68,34.85,25.62,5.06,5.42
humidity,14.26,99.98,85.72,71.48,22.26,13.33
ph,3.50,9.94,6.43,6.47,0.77,1.00
rainfall,20.21,298.56,278.35,103.46,54.96,43.29


`K` spans 200 units; `ph` spans about 6.4. A model that measures the distance
between two fields by adding up per-feature differences — k-nearest neighbours,
an SVM — is therefore, before any learning happens, treating potassium as
roughly **31 times** more important than acidity, purely because of the units
each was recorded in. Nothing about agronomy says that; it is an artifact of
millimetres, milligrams and a logarithmic pH scale sitting in the same table.

The same asymmetry hurts models fitted by gradient descent (logistic regression,
neural networks): a feature with a large numeric range produces large gradients,
so the optimiser zig-zags along the wide directions and crawls along the narrow
ones, converging slowly or not at all.

**Standardising** removes the artifact: every feature is re-expressed in
standard deviations from its own mean, so a difference of "one unit" means the
same thing in every column.

Two model families genuinely do not care — decision trees and their ensembles
(random forest, gradient boosting). A tree only ever asks *"is `K` above 92.5?"*,
and the answer to that question is identical before and after any monotone
rescaling. Week 5 compares both families, and both will be fed the same scaled
data so the comparison measures the models rather than their inputs.

## 2. Encoding the target

`label` is a column of strings. scikit-learn estimators fit on numeric arrays,
so the 22 crop names have to become 22 integers. `LabelEncoder` does exactly
that, and nothing more: it sorts the distinct values alphabetically and assigns
them `0, 1, 2, ...` in that order.

In [4]:
label_encoder = LabelEncoder()
crops["label_encoded"] = label_encoder.fit_transform(crops[TARGET_COLUMN])

mapping = pd.DataFrame(
    {"code": range(len(label_encoder.classes_)), "crop": label_encoder.classes_}
)
mapping.T

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
code,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
crop,apple,banana,blackgram,chickpea,coconut,coffee,cotton,grapes,jute,kidneybeans,...,mango,mothbeans,mungbean,muskmelon,orange,papaya,pigeonpeas,pomegranate,rice,watermelon


In [5]:
# The encoding must agree with the label set frozen in Week 1 - a new, missing
# or misspelled crop has to fail here rather than quietly shift every code.
assert set(label_encoder.classes_) == EXPECTED_LABELS
assert list(label_encoder.classes_) == sorted(EXPECTED_LABELS)

# Encoding is lossless: the codes decode back to exactly the original names.
decoded = label_encoder.inverse_transform(crops["label_encoded"])
assert (decoded == crops[TARGET_COLUMN]).all()

print(len(label_encoder.classes_), "classes, codes", crops["label_encoded"].min(),
      "to", crops["label_encoded"].max())

22 classes, codes 0 to 21


Two things about those integers.

**They are labels, not quantities.** `apple` is 0 and `banana` is 1, but banana
is not "one more than" apple, and `coconut` (5) is not the average of `apple` (0)
and `cotton` (10). The codes are names written in digits. A *classifier* treats
them that way; feeding the same codes to a *regressor* would be a real mistake,
and is why Week 1 insisted the target is categorical and unordered.

**One-hot encoding is not needed here.** A categorical *input* column usually has
to be one-hot encoded, precisely so a model cannot read a false ordering into it.
The *target* of a classifier does not: scikit-learn's classifiers accept integer
class codes (indeed they accept the raw strings too, encoding them internally).
Encoding explicitly keeps the mapping visible and under our control rather than
hidden inside each estimator.

## 3. The stratified train/test split

The split is the moment the test set becomes off-limits. Everything after this
cell that learns anything — a scaler's mean, a model's coefficients, a threshold
chosen by hand — must learn it from `train` only.

`stratified_split()` wraps `train_test_split(..., stratify=y)` with the
project's fixed defaults: 20% held back and `random_state=42`.

In [6]:
train, test = stratified_split(crops)
print("train", train.shape, "| test", test.shape)
print("held back:", round(100 * len(test) / len(crops), 1), "%")

train (1760, 9) | test (440, 9)
held back: 20.0 %


### Why stratify?

An unstratified split shuffles all 2,200 rows and cuts. On average each crop
still lands 80/20, but *on average* is not *on this run*: with 22 classes the
chance that at least one crop is badly over- or under-represented in the test
set is high. Comparing the two side by side makes the difference visible.

In [7]:
from sklearn.model_selection import train_test_split

plain_train, plain_test = train_test_split(
    crops, test_size=DEFAULT_TEST_SIZE, random_state=DEFAULT_RANDOM_STATE
)

comparison = pd.DataFrame(
    {
        "stratified_test": test[TARGET_COLUMN].value_counts(),
        "unstratified_test": plain_test[TARGET_COLUMN].value_counts(),
    }
).sort_index()
comparison.T

label,apple,banana,blackgram,chickpea,coconut,coffee,cotton,grapes,jute,kidneybeans,...,mango,mothbeans,mungbean,muskmelon,orange,papaya,pigeonpeas,pomegranate,rice,watermelon
stratified_test,20,20,20,20,20,20,20,20,20,20,...,20,20,20,20,20,20,20,20,20,20
unstratified_test,23,21,20,26,27,17,17,14,23,20,...,19,24,19,17,14,23,23,23,19,19


In [8]:
print("stratified   test rows per crop: min", comparison["stratified_test"].min(),
      "max", comparison["stratified_test"].max())
print("unstratified test rows per crop: min", comparison["unstratified_test"].min(),
      "max", comparison["unstratified_test"].max())

stratified   test rows per crop: min 20 max 20
unstratified test rows per crop: min 11 max 27


The stratified split gives every crop exactly 20 test rows. The unstratified one
does not, and the crops it short-changes get their per-class scores measured on
fewer rows — so a difference in recall between two crops would partly reflect
how the shuffle fell rather than how the model behaves. With 22 classes and only
440 test rows, that noise is not affordable.

Stratification matters *more*, not less, when classes are imbalanced: a class
with 12 rows can vanish from the test set entirely under a plain shuffle, making
its score undefined.

### Checking the split really is stratified

In [9]:
shares = pd.DataFrame(
    {
        "train": class_proportions(train),
        "test": class_proportions(test),
    }
)
shares["difference"] = (shares["train"] - shares["test"]).abs()
print("largest difference in class share:", shares["difference"].max())
shares.head(5).round(6)

largest difference in class share: 0.0


,train,test,difference
label,,,
apple,0.045455,0.045455,0.0
banana,0.045455,0.045455,0.0
blackgram,0.045455,0.045455,0.0
chickpea,0.045455,0.045455,0.0
coconut,0.045455,0.045455,0.0


Every crop holds 1/22 = 4.5454...% of both halves, so the largest difference in
proportion is zero to within floating-point noise. This is the Week 2 class
balance, preserved.

### Why a fixed random seed

`train_test_split` shuffles, and a shuffle needs randomness. Unseeded, it draws a
different split on every run: two students running identical code would get
different accuracies in Week 4, and neither could tell an improvement from
noise. `random_state=42` makes the shuffle deterministic — the same rows land in
the same half, forever.

In [10]:
again_train, again_test = stratified_split(crops)
print("same split on a second call:", again_test.equals(test))

other_train, other_test = stratified_split(crops, random_state=7)
overlap = len(set(map(tuple, other_test[FEATURES].to_numpy()))
              & set(map(tuple, test[FEATURES].to_numpy())))
print("rows shared with the seed-7 test set:", overlap, "of", len(test))

same split on a second call: True
rows shared with the seed-7 test set: 84 of 440


The seed's *value* means nothing; 42 is a convention. What matters is that it is
fixed and recorded. It must never be chosen to make a score look better —
searching seeds for the best test accuracy is overfitting the test set by hand,
and the resulting number would not survive contact with real fields.

## 4. Scaling: fit on train, transform both

`build_preprocessor()` returns an **unfitted** `ColumnTransformer` that routes
the seven numeric features through a `StandardScaler` and drops everything else
(including the target, which must never travel through the feature
preprocessor).

Three method names, and the distinction between them is the week's central idea:

| Call | What it does | Where it is allowed |
| --- | --- | --- |
| `fit(X)` | Learns and stores parameters — here, each column's mean and std | Training data only |
| `transform(X)` | Applies the stored parameters | Any data: train, test, or one row from an API in Week 10 |
| `fit_transform(X)` | Both, in one call | Training data only |

In [11]:
preprocessor = build_preprocessor()                  # unfitted
X_train_scaled = preprocessor.fit_transform(train)   # LEARNS from train, then applies
X_test_scaled = preprocessor.transform(test)         # APPLIES train's statistics
print(type(X_train_scaled).__name__, X_train_scaled.shape, "|", X_test_scaled.shape)

ndarray (1760, 7) | (440, 7)


In [12]:
scaler = preprocessor.named_transformers_["numeric"]
names = list(preprocessor.get_feature_names_out())
learned = pd.DataFrame(
    {"mean_": scaler.mean_, "scale_ (std)": scaler.scale_}, index=names
)
learned.round(3)

,mean_,scale_ (std)
N,50.548,36.852
P,53.340,32.938
K,48.143,50.694
temperature,25.609,5.079
humidity,71.417,22.274
ph,6.474,0.783
rainfall,103.452,54.977


Those 14 numbers are the entire fitted state of the preprocessor, and every one
of them was computed from the 1,760 training rows. They are what `transform`
subtracts and divides by, whatever data it is later handed.

In [13]:
def moments(array, columns):
    """Column means and population standard deviations, as a small frame."""
    return pd.DataFrame(
        {"mean": array.mean(axis=0), "std": array.std(axis=0, ddof=0)}, index=columns
    )


pd.concat(
    {"train (fitted)": moments(X_train_scaled, names),
     "test (transformed)": moments(X_test_scaled, names)},
    axis=1,
).round(3)

train (fitted)      test (transformed)       
                      mean  std               mean    std
N                      0.0  1.0              0.001  1.008
P                      0.0  1.0              0.003  1.006
K                      0.0  1.0              0.001  0.994
temperature           -0.0  1.0              0.007  0.984
humidity              -0.0  1.0              0.015  0.996
ph                    -0.0  1.0             -0.028  0.939
rainfall               0.0  1.0              0.001  0.997

Read that table carefully, because it is the whole lesson:

* **Train**: mean 0.000 and std 1.000 in every column, exactly. That is what
  fitting *means* — the scaler subtracted these rows' own mean and divided by
  these rows' own standard deviation.
* **Test**: close to 0 and 1, but not equal to them. It *should not* be equal.
  The test rows were shifted by the training mean, which is not quite their own
  mean. The small residual is the honest measure of how much the held-out rows
  differ from the training rows — exactly the difference a real deployment will
  also see.

If the test column read 0.000/1.000 too, the scaler would have been fitted on
the test rows, and the model's later score would be flattered by information it
will not have in production.

### What the leak would have looked like

The tempting shortcut — scale everything, then split — is one line shorter and
quietly wrong.

In [14]:
leaky = build_preprocessor().fit(crops)      # fitted on ALL 2,200 rows: WRONG
leaky_scaler = leaky.named_transformers_["numeric"]

drift = pd.DataFrame(
    {
        "train-only mean": scaler.mean_,
        "all-data mean": leaky_scaler.mean_,
        "difference": leaky_scaler.mean_ - scaler.mean_,
    },
    index=names,
)
drift.round(4)

,train-only mean,all-data mean,difference
N,50.5477,50.5518,0.0041
P,53.3398,53.3627,0.0230
K,48.1432,48.1491,0.0059
temperature,25.6094,25.6162,0.0068
humidity,71.4168,71.4818,0.0650
ph,6.4738,6.4695,-0.0044
rainfall,103.4516,103.4637,0.0121


The differences are small here — the split is stratified and this dataset is
unusually homogeneous, so 2,200 rows and 1,760 rows have nearly the same means.
That is precisely why the habit has to be procedural rather than judged case by
case: on this dataset the leak barely moves a number, on a dataset with a heavy
tail or a rare class it can move a lot, and there is no way to know which you
have without already having leaked.

The rule is therefore mechanical: **`fit_transform` on train, `transform` on
everything else.** Never `fit` anything on data you intend to evaluate on.

## 5. The `Pipeline` object

A `ColumnTransformer` handles *columns*; a `Pipeline` handles *steps*, chaining
them into one object with a single `fit`/`transform`/`predict` surface.

Right now the pipeline has one step and does nothing extra. It is introduced now
because from Week 4 onward every model is appended to this same object rather
than bolted on beside it:

```python
Pipeline([("preprocess", build_preprocessor()), ("model", LogisticRegression())])
```

At that point `pipeline.fit(X_train, y_train)` fits the scaler *and* the model
in one call, `pipeline.predict(X_new)` scales *and* predicts, and the
cross-validation of Week 6 re-fits the scaler inside every fold — which is the
only way to cross-validate a scaled model without leaking each validation fold
into its own training. The single object that gets saved in Week 9 and served by
the API in Week 10 is this one.

In [15]:
pipeline = build_preprocessing_pipeline()
pipeline.fit(train)

# Identical numbers to the bare ColumnTransformer - the wrapper adds structure,
# not behaviour.
print("pipeline == preprocessor output:",
      np.allclose(pipeline.transform(test), X_test_scaled))
pipeline

pipeline == preprocessor output: True


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['N', 'P', 'K', 'temperature',
                                                   'humidity', 'ph',
                                                   'rainfall'])],
                                   verbose_feature_names_out=False))])

## 6. Saving the prepared data

Week 4 should not have to re-run this notebook to get a training set. The splits
are therefore written to `data/processed/`, which the repository layout reserves
for anything derived from the raw file. `data/raw/` is untouched, as always.

Five files, all deterministic given the fixed seed:

| File | Contents |
| --- | --- |
| `train.csv` | 1,760 rows: the seven raw features, `label`, and `label_encoded` |
| `test.csv` | 440 rows, same columns |
| `train_scaled.csv`, `test_scaled.csv` | The same rows with features standardised by the train-fitted scaler, plus `label_encoded` |
| `label_classes.csv` | The code-to-crop mapping, so the integers can always be read back |

CSV rather than `.npy` because these tables are small (a few hundred KB), and a
CSV can be opened, diffed and inspected by a human — worth far more here than
the speed of a binary format. The unscaled `train.csv`/`test.csv` are the
canonical artifacts: scaling is cheap to redo and, being part of the pipeline,
*should* be redone inside cross-validation rather than baked in. The scaled
copies exist so that Week 4 can load model-ready arrays directly.

In [16]:
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS_TO_SAVE = [*FEATURES, TARGET_COLUMN, "label_encoded"]
train[COLUMNS_TO_SAVE].to_csv(PROCESSED_DIR / "train.csv", index=False)
test[COLUMNS_TO_SAVE].to_csv(PROCESSED_DIR / "test.csv", index=False)

scaled_train = pd.DataFrame(X_train_scaled, columns=names)
scaled_train["label_encoded"] = train["label_encoded"].to_numpy()
scaled_test = pd.DataFrame(X_test_scaled, columns=names)
scaled_test["label_encoded"] = test["label_encoded"].to_numpy()
scaled_train.to_csv(PROCESSED_DIR / "train_scaled.csv", index=False)
scaled_test.to_csv(PROCESSED_DIR / "test_scaled.csv", index=False)

mapping.to_csv(PROCESSED_DIR / "label_classes.csv", index=False)

for path in sorted(PROCESSED_DIR.glob("*.csv")):
    print(f"{path.name:20s} {path.stat().st_size / 1024:6.1f} KB")

label_classes.csv       0.2 KB
test.csv               29.9 KB
test_scaled.csv        60.4 KB
train.csv             119.9 KB
train_scaled.csv      241.6 KB


In [17]:
# Reload check: what was written is what comes back.
reloaded_train = pd.read_csv(PROCESSED_DIR / "train.csv")
reloaded_classes = pd.read_csv(PROCESSED_DIR / "label_classes.csv")

assert reloaded_train.shape == (1_760, len(COLUMNS_TO_SAVE))
assert list(reloaded_classes["crop"]) == list(label_encoder.classes_)
assert set(reloaded_train[TARGET_COLUMN]) == EXPECTED_LABELS
assert reloaded_train[TARGET_COLUMN].value_counts().nunique() == 1  # 80 rows per crop

reloaded_train.head(3).round(2)

,N,P,K,temperature,humidity,ph,rainfall,label,label_encoded
0,0,18,14,29.77,92.01,7.21,114.42,orange,16
1,9,122,201,29.59,80.92,5.57,68.06,grapes,7
2,11,71,24,21.14,22.72,5.61,141.61,kidneybeans,9


## 7. What this week produced, and what it deliberately did not

**Produced**

* A target encoded as 22 integers, with a lossless mapping saved beside it.
* A stratified 80/20 split — 1,760 training rows and 440 test rows, exactly 80
  and 20 per crop — reproducible from `random_state=42`.
* A `ColumnTransformer` fitted **on the training rows only**, whose 14 learned
  numbers (seven means, seven standard deviations) transform train and test
  alike, leaving the training features at mean 0 / std 1 and the test features
  near but not equal to it.
* Those splits on disk in `data/processed/`.

**Not produced, on purpose**

* No model, no accuracy, no comparison of algorithms — Week 4.
* No feature engineering: no new columns, no interactions, no log transforms.
  The seven features go forward as they are; the skew Week 2 found in `K` and
  `rainfall` is still there, because standardising is a linear rescaling and
  does not remove skew.
* No rows removed. The values the IQR rule flagged in Week 2 are whole crop
  populations, and they are still here.
* No look at the test set beyond counting its rows and its class shares. Nothing
  fitted in this notebook has seen a test value.

Week 4 loads `data/processed/train.csv`, fits the first baseline, and finally
gets to ask how hard this problem actually is.